# Validate Keyword Labeling

**Purpose:** Determine whether keyword-based classification on clean product name/generic name text
is accurate enough to trust for auto-labeling unlabeled data. This notebook measures accuracy only —
it does NOT write any labels, train any models, or run any pseudo-labeling.

**Prerequisites:**
- Labeled manifest at `backend/data/raw/manifest.csv` (produced by `build_dataset.py`)
- HuggingFace dataset `openfoodfacts/product-database` (split `beauty`) accessible for text extraction
- `app/services/extraction.py` with `classify_category()` and `CATEGORY_KEYWORDS`

In [1]:
import sys, os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from dotenv import load_dotenv

BACKEND_DIR = Path.cwd().parent
sys.path.insert(0, str(BACKEND_DIR))
load_dotenv(BACKEND_DIR / ".env")

from app.services.extraction import classify_category, CATEGORY_KEYWORDS

MANIFEST_PATH = BACKEND_DIR / "data" / "raw" / "manifest.csv"
print(f"Manifest: {MANIFEST_PATH}")
print(f"Keyword categories: {list(CATEGORY_KEYWORDS.keys())}")
print(f"Keywords per class: {{k: len(v) for k, v in CATEGORY_KEYWORDS.items()}}")

Manifest: c:\Projects\cosmetic-expiry-scanner\backend\data\raw\manifest.csv
Keyword categories: ['skincare', 'haircare', 'makeup']
Keywords per class: {k: len(v) for k, v in CATEGORY_KEYWORDS.items()}


In [2]:
def extract_text(field):
    """Extract English text from STRUCT(lang, text)[] field, fallback to first available."""
    if not isinstance(field, list) or len(field) == 0:
        return None
    for entry in field:
        if isinstance(entry, dict):
            lang = entry.get("lang", "")
            text = entry.get("text", "")
            if lang.startswith("en") and text:
                return text
    for entry in field:
        if isinstance(entry, dict) and entry.get("text"):
            return entry["text"]
    return None


manifest = pd.read_csv(MANIFEST_PATH, dtype={"code": str})
manifest = manifest.rename(columns={"label": "true_category"})
codes_needed = set(manifest["code"])
print(f"Manifest: {len(manifest):,} rows")
print(f"Unique codes: {len(codes_needed):,}")
print(f"Sample manifest codes: {list(codes_needed)[:5]}")
print(f"\nClass distribution:")
print(manifest["true_category"].value_counts().to_string())

Manifest: 11,329 rows
Unique codes: 11,329
Sample manifest codes: ['8710447484265', '4020829008793', '7640183490651', '3323037218635', '4088700042038']

Class distribution:
true_category
skincare    8013
haircare    2497
makeup       819


In [3]:
from datasets import load_dataset
from tqdm import tqdm

ds = load_dataset("openfoodfacts/product-database", split="beauty", streaming=True)

text_data = {}
remaining = set(codes_needed)

for row in tqdm(ds, desc="Fetching product names from HF"):
    code = str(row["code"])
    if code in remaining:
        product_name = extract_text(row.get("product_name"))
        generic_name = extract_text(row.get("generic_name"))
        text_data[code] = {
            "product_name_text": product_name,
            "generic_name_text": generic_name,
        }
        remaining.discard(code)
        if not remaining:
            break

print(f"\nFound text for {len(text_data):,} / {len(codes_needed):,} codes")
if remaining:
    print(f"Missing: {len(remaining):,} codes (will be dropped from analysis)")

text_df = pd.DataFrame.from_dict(text_data, orient="index").reset_index()
text_df = text_df.rename(columns={"index": "code"})

df = manifest.merge(text_df, on="code", how="inner")
print(f"\nMerged DataFrame: {len(df):,} rows")
print(f"product_name non-null: {df['product_name_text'].notna().sum():,} ({df['product_name_text'].notna().mean()*100:.1f}%)")
print(f"generic_name non-null: {df['generic_name_text'].notna().sum():,} ({df['generic_name_text'].notna().mean()*100:.1f}%)")
print(f"\nClass distribution in merged set:")
print(df["true_category"].value_counts().to_string())

Fetching product names from HF: 74218it [00:44, 1651.77it/s]


Found text for 11,329 / 11,329 codes

Merged DataFrame: 11,329 rows
product_name non-null: 10,246 (90.4%)
generic_name non-null: 2,345 (20.7%)

Class distribution in merged set:
true_category
skincare    8013
haircare    2497
makeup       819


In [4]:
# Contamination check: does the labeled skincare set include non-cosmetic
# products (toothpaste, mouthwash, dish soap) picked up by an overly broad
# OBF tag in CATEGORY_MAP?
contamination_terms = ["dentifrice", "toothpaste", "mouthwash", "bain de bouche",
                       "dish soap", "toilettenpapier", "toilet paper"]

skincare_labeled = df[df["true_category"] == "skincare"]
hits = skincare_labeled[skincare_labeled["product_name_text"]
                        .str.lower()
                        .str.contains("|".join(contamination_terms), na=False)]
print(f"{len(hits)} potentially contaminated skincare labels out of {len(skincare_labeled)}")
print(hits[["code", "product_name_text"]].head(20))

365 potentially contaminated skincare labels out of 8013
              code                                  product_name_text
53   0773792450406                           Dentifrice pour enfants 
71        15595635                                         Dentifrice
116  3014260096878               Dentifrice Bi-fluoré Dents Sensibles
179  3094905001085                 Pro-Émail Bain de bouche quotidien
226  3178040334229                    Dentifrice au fluor goût fraise
249  3178040673335                    Bain de bouche EXPERT COMPLET 7
292  3178041308793  Teraxyl - 2 en 1 - Dentifrice + Bain de Bouche...
352  3222472861171  Dentifrice 2 en 1 Blancheur formule micro-part...
355  3222474977962                      Dentifrice au fluor Fraîcheur
360  3222476255013                            Dentifrice menthe douce
401  3250392228415                               Dentifrice Blancheur
436  3256222741192                                 Dentifrice Plantes
498  3263855909424               

In [5]:
# Find which OBF tag is causing these toothpaste/mouthwash products
# to be classified as skincare in CATEGORY_MAP
from collections import Counter

contaminated_codes = set(hits["code"])
print(f"Looking up categories_tags for {len(contaminated_codes)} contaminated codes...")

ds2 = load_dataset("openfoodfacts/product-database", split="beauty", streaming=True)

tag_counter = Counter()
remaining_codes = set(contaminated_codes)
examples_by_tag = {}

for row in tqdm(ds2, desc="Scanning for contaminated codes"):
    code = str(row["code"])
    if code in remaining_codes:
        tags = row.get("categories_tags") or []
        tag_counter.update(tags)
        for t in tags:
            examples_by_tag.setdefault(t, []).append(code)
        remaining_codes.discard(code)
        if not remaining_codes:
            break

print(f"\nMost common tags across contaminated products:")
for tag, count in tag_counter.most_common(20):
    print(f"  {tag}: {count}")

Looking up categories_tags for 365 contaminated codes...


Scanning for contaminated codes: 74131it [00:41, 1791.83it/s]


Most common tags across contaminated products:
  en:hygiene: 361
  en:toothpastes: 280
  en:mouthwash: 71
  en:open-beauty-facts: 55
  en:non-food-products: 53
  en:whitening-toothpastes: 22
  en:sensitivity-toothpastes: 14
  en:children-s-toothpastes: 8
  en:tartar-control-toothpastes: 7
  fr:dentifrices-pour-enfants: 6
  en:Oral care: 6
  en:mouthwashes: 6
  en:Cosmetics: 5
  en:accessories: 2
  en:moist-wipes: 2
  en:Fluoride toothpastes: 2
  en:Dental care: 2
  fr:non-food-products: 1
  fr:open-beauty-facts: 1
  fr:dentifrice-antibacterien: 1


In [ ]:
df["pred_name"] = df["product_name_text"].apply(
    lambda t: classify_category(t)[0] if pd.notna(t) and isinstance(t, str) else None
)
df["pred_generic"] = df["generic_name_text"].apply(
    lambda t: classify_category(t)[0] if pd.notna(t) and isinstance(t, str) else None
)
df["pred_combined"] = df["pred_name"].fillna(df["pred_generic"])

print("product_name predictions:")
print(df["pred_name"].value_counts(dropna=False).to_string())
print(f"\ngeneric_name predictions:")
print(df["pred_generic"].value_counts(dropna=False).to_string())
print(f"\ncombined predictions:")
print(df["pred_combined"].value_counts(dropna=False).to_string())

In [ ]:
# Coverage diagnostic: what do unmatched products look like?
no_match = df[df["pred_combined"].isna()].copy()
print(f"No keyword match: {len(no_match):,} / {len(df):,} ({len(no_match)/len(df)*100:.1f}%)")
print(f"\nClass distribution of unmatched products:")
print(no_match["true_category"].value_counts().to_string())

# Check how many have non-null product_name (text exists but no keyword matched)
has_name = no_match["product_name_text"].notna().sum()
has_generic = no_match["generic_name_text"].notna().sum()
print(f"\nOf unmatched: {has_name:,} have product_name, {has_generic:,} have generic_name")

# Sample product names from unmatched rows
print(f"\n{'='*60}")
print("  SAMPLE UNMATCHED PRODUCTS (40 random)")
print(f"{'='*60}")
sample = no_match[["code", "true_category", "product_name_text", "generic_name_text"]].sample(40, random_state=42)
for _, row in sample.iterrows():
    name = str(row["product_name_text"])[:90] if pd.notna(row["product_name_text"]) else "(none)"
    generic = str(row["generic_name_text"])[:90] if pd.notna(row["generic_name_text"]) else "(none)"
    print(f"\n  [{row['true_category']}] {name}")
    if generic != "(none)":
        print(f"    generic: {generic}")

In [ ]:
classes = sorted(df["true_category"].unique())


def evaluate_approach(df, pred_col, approach_name):
    total = len(df)
    has_prediction = df[pred_col].notna()
    coverage_count = has_prediction.sum()
    coverage_pct = coverage_count / total * 100

    scored = df[has_prediction].copy()
    if len(scored) == 0:
        print(f"\n{'='*60}")
        print(f"  {approach_name}: NO PREDICTIONS MADE")
        print(f"{'='*60}")
        return {"coverage_pct": 0, "accuracy_pct": 0, "per_class_min": 0}

    correct = (scored[pred_col] == scored["true_category"]).sum()
    accuracy_pct = correct / len(scored) * 100

    print(f"\n{'='*60}")
    print(f"  {approach_name}")
    print(f"{'='*60}")
    print(f"Coverage: {coverage_count}/{total} ({coverage_pct:.1f}%)")
    print(f"Accuracy: {correct}/{len(scored)} ({accuracy_pct:.1f}%)")
    print()
    print(classification_report(
        scored["true_category"], scored[pred_col],
        labels=classes, zero_division=0
    ))

    cm = confusion_matrix(scored["true_category"], scored[pred_col], labels=classes)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=classes, yticklabels=classes)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {approach_name}")
    plt.tight_layout()
    plt.show()

    per_class_acc = {}
    for cls in classes:
        mask = scored["true_category"] == cls
        if mask.sum() > 0:
            per_class_acc[cls] = (scored.loc[mask, pred_col] == cls).mean() * 100
        else:
            per_class_acc[cls] = 0
    per_class_min = min(per_class_acc.values()) if per_class_acc else 0

    return {
        "coverage_pct": coverage_pct,
        "accuracy_pct": accuracy_pct,
        "per_class_min": per_class_min,
        "per_class_acc": per_class_acc,
    }


results = {}
for pred_col, name in [
    ("pred_name", "product_name only"),
    ("pred_generic", "generic_name only"),
    ("pred_combined", "combined (name → generic fallback)"),
]:
    results[name] = evaluate_approach(df, pred_col, name)

---
## Go / No-Go Summary

Replace the placeholder values below after running the cells above.

```
Coverage (name):             XX.X%
Coverage (generic):          XX.X%
Coverage (combined):         XX.X%

Accuracy (product_name):     XX.X%
Accuracy (generic_name):     XX.X%
Accuracy (combined):         XX.X%

Per-class minimum (combined): XX.X% [skincare: XX.X%, haircare: XX.X%, makeup: XX.X%]

Recommendation: [PROCEED if best approach >= 85% accuracy AND coverage >= 50% AND per-class min >= 70%]
                [DO NOT PROCEED if below threshold — explain gap]
```

In [ ]:
best = max(results, key=lambda k: results[k]["accuracy_pct"])
best_r = results[best]

ACC_THRESHOLD = 85.0
COVERAGE_THRESHOLD = 50.0
PER_CLASS_THRESHOLD = 70.0

print("=" * 60)
print("  GO / NO-GO SUMMARY")
print("=" * 60)
print()
for name, r in results.items():
    print(f"  {name}:")
    print(f"    Coverage:  {r['coverage_pct']:.1f}%")
    print(f"    Accuracy:  {r['accuracy_pct']:.1f}%")
    if "per_class_acc" in r:
        pca = r["per_class_acc"]
        print(f"    Per-class: ", end="")
        print(", ".join(f"{k}: {v:.1f}%" for k, v in sorted(pca.items())))
    print()

print(f"Best approach: {best}")
print(f"  Accuracy:     {best_r['accuracy_pct']:.1f}% (threshold: {ACC_THRESHOLD}%)")
print(f"  Coverage:     {best_r['coverage_pct']:.1f}% (threshold: {COVERAGE_THRESHOLD}%)")
print(f"  Per-class min: {best_r['per_class_min']:.1f}% (threshold: {PER_CLASS_THRESHOLD}%)")
print()

passes_acc = best_r["accuracy_pct"] >= ACC_THRESHOLD
passes_cov = best_r["coverage_pct"] >= COVERAGE_THRESHOLD
passes_cls = best_r["per_class_min"] >= PER_CLASS_THRESHOLD

if passes_acc and passes_cov and passes_cls:
    print("  >>> RECOMMENDATION: PROCEED")
    print("  Keyword matching clears all thresholds for auto-labeling.")
else:
    print("  >>> RECOMMENDATION: DO NOT PROCEED")
    gaps = []
    if not passes_acc:
        gaps.append(f"accuracy {best_r['accuracy_pct']:.1f}% < {ACC_THRESHOLD}%")
    if not passes_cov:
        gaps.append(f"coverage {best_r['coverage_pct']:.1f}% < {COVERAGE_THRESHOLD}%")
    if not passes_cls:
        gaps.append(f"per-class min {best_r['per_class_min']:.1f}% < {PER_CLASS_THRESHOLD}%")
    print(f"  Gaps: {'; '.join(gaps)}")
    print("  Refine keyword list or consider alternative labeling approach.")

In [ ]:
best_pred_col = {
    "product_name only": "pred_name",
    "generic_name only": "pred_generic",
    "combined (name → generic fallback)": "pred_combined",
}[best]

scored = df[df[best_pred_col].notna()].copy()
errors = scored[scored[best_pred_col] != scored["true_category"]].copy()

print(f"Misclassified: {len(errors):,} / {len(scored):,} scored rows "
      f"({len(errors)/len(scored)*100:.1f}%)")
print(f"\nErrors by true class:")
print(errors["true_category"].value_counts().to_string())
print(f"\nErrors by predicted class:")
print(errors[best_pred_col].value_counts().to_string())
print(f"\nConfusion pairs (true → predicted):")
pair_counts = (
    errors.groupby(["true_category", best_pred_col])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print(pair_counts.to_string(index=False))

print(f"\n{'='*60}")
print("  SAMPLE MISCLASSIFICATIONS (up to 30)")
print(f"{'='*60}")
display_cols = ["code", "true_category", best_pred_col, "product_name_text", "generic_name_text"]
sample = errors[display_cols].head(30)
for _, row in sample.iterrows():
    print(f"\n  code: {row['code']}")
    print(f"  true: {row['true_category']}  |  predicted: {row[best_pred_col]}")
    name = row["product_name_text"]
    generic = row["generic_name_text"]
    if pd.notna(name):
        print(f"  name:    {str(name)[:120]}")
    if pd.notna(generic):
        print(f"  generic: {str(generic)[:120]}")